# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane:** Lane 2: Refresh / Content Opportunity Scoring.

**Why this lane:** content teams cannot review thousands of pages, but some of those pages are quietly losing search visibility. Picking the right few pages each week is a real, repeated decision, and the starter data has the signals to study it: impressions, clicks, position, CTR, page age and update history. The goal is a ranked weekly review list that an editor can trust, built only from signals that exist before the decision.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Starter dataset: {len(df):,} pages x {df.shape[1]} columns | {df['client_id'].nunique()} pseudonymized clients")
print("Trend of each page (impressions, last 30d vs previous 30d):")
print(df["trend_direction"].value_counts().to_string())

Starter dataset: 30,000 pages x 44 columns | 32 pseudonymized clients
Trend of each page (impressions, last 30d vs previous 30d):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

* **Research question:** which pages should an editor review first, using only signals known before the decision?
* **Unit:** one page (`content_id`).
* **Decision it improves:** which ~50 pages go into this week's review batch, instead of picking at random.
* **Who acts:** the editor / content team.
* **Action:** review the page, then update facts and links, rewrite the title and meta description, or leave it alone.
* **Cost of a wrong call:**
  * **False alarm** (a healthy page in the batch): wasted editor time.
  * **Miss** (a declining page left out): a visible page keeps losing impressions for another week.

In [2]:
# One row = one page? And how many pages does each client have (the editor's real workload)?
print(f"Rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} -> one row = one page: {len(df) == df['content_id'].nunique()}")
per_client = df.groupby("client_id").size()
print(f"Pages per client: median {per_client.median():.0f}, largest {per_client.max():,}")
print(f"A weekly review batch of 50 pages covers {50 / per_client.median():.0%} of a median client's pages "
      f"and {50 / per_client.max():.1%} of the largest client's pages -> picking the right 50 matters.")

Rows: 30,000 | unique content_id: 30,000 -> one row = one page: True
Pages per client: median 567, largest 7,008
A weekly review batch of 50 pages covers 9% of a median client's pages and 0.7% of the largest client's pages -> picking the right 50 matters.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three real numbers (printed by the cell below):
* **54.2% of pages are declining:** impressions fell by more than 20% (last 30 days vs the previous 30). The problem is widespread, and 54.2% is also the base rate any ranking has to beat.
* **r = 0.001 between keyword search volume and real impressions:** keyword volume alone says almost nothing about how visible a page actually is.
* **55.8% of pages have ≥ 500 impressions in 90 days:** most pages have enough search data to be judged.

In [3]:
# Three real numbers that make the lane worth 7 more weeks
declining = (df["trend_direction"] == "down").mean()
r_vol_imp = df["search_volume"].corr(df["impressions_90d"])
visible = (df["impressions_90d"] >= 500).mean()

print(f"1. Pages with impressions down >20% (last 30d vs previous 30d): {declining:.1%}")
print(f"2. Correlation, keyword search volume vs 90-day impressions:    r = {r_vol_imp:.3f}")
print(f"3. Pages with at least 500 impressions in 90 days:              {visible:.1%}")

1. Pages with impressions down >20% (last 30d vs previous 30d): 54.2%
2. Correlation, keyword search volume vs 90-day impressions:    r = 0.001
3. Pages with at least 500 impressions in 90 days:              55.8%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

* **I can say:** which signals are *observed* together with decline, *measured* on held-out clients, and *directional*. The output is a *decision-support* ranking for a human editor.
* **I cannot say:**
  * that refreshing a page *causes* recovery: there is no experiment or control group in this data;
  * anything about how Google's algorithm works.
* **Never inputs:** `trend_direction`, `trend_pct`, and every `_last_30d` / `_prev_30d` column, because the label is computed from them.

In [4]:
# Careful words, backed by the data: what the columns can and cannot support
label_cols = ["trend_direction", "trend_pct"]
outcome_window = [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]
print("Label comes from:", label_cols, "-> these can never be inputs")
print("Columns that share the label's 30-day windows (never inputs):", outcome_window)
print("Experiment / treatment columns (needed for causal claims):",
      [c for c in df.columns if "treat" in c or "experiment" in c] or "none -> observational only, no causal claims")

Label comes from: ['trend_direction', 'trend_pct'] -> these can never be inputs
Columns that share the label's 30-day windows (never inputs): ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
Experiment / treatment columns (needed for causal claims): none -> observational only, no causal claims


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.